# 04 - Componente Agentic / RAG

Agente de IA para analisis de churn y recomendaciones de retencion

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import joblib
from src.agentic.rag_engine import ChurnRAGEngine
from src.agentic.agent import ChurnAgent

In [2]:
# 1. Inicializar motor RAG con datos historicos
rag = ChurnRAGEngine('../data/raw/telco_customer_churn.csv')

Documentos creados: 7043


In [3]:
# 2. Ver estadisticas de churn
stats = rag.get_churn_stats()
for k, v in stats.items():
    if isinstance(v, float):
        print(f'{k}: {v:.4f}')
    else:
        print(f'{k}: {v}')

total_clientes: 7043
churn_rate: 0.3085
avg_tenure_churn: 34.4837
avg_tenure_no_churn: 37.1386
avg_monthly_churn: 69.0685
avg_monthly_no_churn: 68.1930
top_contract_churn: Month-to-month


In [4]:
# 3. Cargar modelo entrenado
model = joblib.load('../models/best_model.pkl')
print(f'Modelo cargado: {type(model).__name__}')

Modelo cargado: LogisticRegression


In [5]:
# 4. Inicializar agente
agent = ChurnAgent(model=model, rag_engine=rag)
print('Agente inicializado.')

Agente inicializado.


In [6]:
# 5. Analisis de un cliente en riesgo
sample_customer = {
    'tenure': 3,
    'MonthlyCharges': 89.50,
    'TotalCharges': 268.50,
    'Contract': 'Month-to-month',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'TechSupport': 'No',
    'PaymentMethod': 'Electronic check',
    'num_services': 3
}

# Simular prediccion
print('=== Analisis de Cliente ===')
explanation = agent.explain_prediction(sample_customer, prediction=1, probability=0.85)
print(explanation)

=== Analisis de Cliente ===
**Analisis del Cliente**

- Riesgo de churn: ALTO (85.0%)
- Antiguedad: 3 meses
- Cargo mensual: $89.50
- Tipo de contrato: Month-to-month
- Servicio internet: Fiber optic

**Factores de Riesgo:**
- Contrato mes a mes (mayor riesgo de abandono)
- Cliente nuevo (< 12 meses)
- Servicio de fibra optica (mayor churn)
- Sin seguridad online
- Sin soporte tecnico
- Pago con cheque electronico


In [7]:
# 6. Recomendacion de retencion
recommendation = agent.generate_retention_recommendation(
    sample_customer, prediction=1, probability=0.85
)
print(recommendation)

**Plan de Retencion** (Riesgo: 85.0%)

Acciones recomendadas:
1. Ofrecer descuento por cambio a contrato anual
2. Ofrecer seguridad online gratis por 3 meses
3. Incluir soporte tecnico premium como beneficio
4. Revisar precio de fibra optica - considerar ajuste
5. Programa de fidelizacion para nuevos clientes


In [8]:
# 7. Analisis de cohort
test_data = pd.read_csv('../data/processed/test.csv')
cohort_analysis = agent.analyze_cohort(test_data)
print('=== Analisis de Cohort (Test Set) ===')
for k, v in cohort_analysis.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.2f}')
    else:
        print(f'  {k}: {v}')

=== Analisis de Cohort (Test Set) ===
  total_customers: 1409
  high_risk: 435
  medium_risk: 0
  low_risk: 974
  risk_percentage: 30.87


In [9]:
# 8. Query al motor RAG
response = agent.query('Cuales son los factores principales de churn?')
print(response)

**Respuesta basada en datos historicos:**

Estadisticas generales:
- Total clientes: 7043
- Churn rate: 30.9%
- Antiguedad promedio (churn): 34.5 meses
- Antiguedad promedio (sin churn): 37.1 meses
- Cargo mensual promedio (churn): $69.07
- Cargo mensual promedio (sin churn): $68.19

Documentos relevantes encontrados: 3
